In [17]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "bcosgnn").is_dir():
            return p
    raise RuntimeError("Could not locate repo root (pyproject.toml + bcosgnn/).")

project_root = find_repo_root(current_dir)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [18]:
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_networkx
from torch_geometric.nn.conv import GINEConv
from torch_geometric.nn.aggr import MeanAggregation, SumAggregation

from tqdm.auto import tqdm

from bcos.modules import BcosLinear
from bcosgnn.explain_edge_attr import explain as explain_edge_attr
import os
import shutil
from torch_geometric.data import InMemoryDataset
import tqdm
import sys
import os

import functools
import itertools
import operator
from typing import Any
import torch
from torch_geometric.data import Dataset, download_url
from torch.utils.data import random_split
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from bcos.modules import BcosLinear, BcosSequential
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation
from torch_geometric.utils import add_self_loops, degree
from torchmetrics import AUROC
from torchmetrics.classification import BinaryAccuracy
from tqdm import tqdm
import networkx as nx
import torch.nn.functional as F
from bcosgnn.explain import explain
from bcosgnn.evaluation import get_attribution_scores

# MolHIV: B-COS GINE vs Vanilla GINE
This notebook trains and compares **B-COS GINE** and **Vanilla GINE** on MolHIV using **3 seeds**.

Reported metrics for each model:
- Test ROC-AUC (mean ± std across seeds)
- Best validation epoch (mean ± std across seeds)

In [19]:
import copy
import random
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.nn.aggr import SumAggregation

from bcos.modules import BcosLinear

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.9.1
CUDA available: False


In [20]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class TrainConfig:
    hidden_dim: int = 64
    num_layers: int = 4
    dropout: float = 0.2
    b: float =3
    max_out: int = 1
    batch_size: int = 64
    max_epochs: int = 120
    lr: float = 1e-3
    weight_decay: float = 1e-5
    early_stop_patience: int = 20

cfg = TrainConfig()
SEEDS = (0, 1, 2)
TEST_SIZE = 0.2
VAL_SIZE = 0.1
DATASET_ROOT = str(project_root / "data" / "MolHIV")

print(cfg)

TrainConfig(hidden_dim=64, num_layers=4, dropout=0.2, b=3, max_out=1, batch_size=64, max_epochs=120, lr=0.001, weight_decay=1e-05, early_stop_patience=20)


In [21]:
def _as_binary_label(y_tensor: torch.Tensor) -> int:
    y = float(y_tensor.view(-1)[0].item())
    return int(y > 0.0)

def load_molhiv_dataset(root: str):
    dataset = MoleculeNet(root=root, name="HIV")
    filtered = []
    for data in dataset:
        y = data.y.view(-1)[0]
        if torch.isfinite(y):
            data.y = torch.tensor([_as_binary_label(data.y)], dtype=torch.long)
            filtered.append(data)
    return filtered

def stratified_split_indices(labels: np.ndarray, seed: int, test_size: float, val_size: float):
    all_idx = np.arange(len(labels))
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(all_idx, labels))

    labels_trainval = labels[trainval_idx]
    rel_val_size = val_size / (1.0 - test_size)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=rel_val_size, random_state=seed)
    train_rel, val_rel = next(sss_val.split(np.zeros_like(labels_trainval), labels_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]
    return train_idx, val_idx, test_idx

dataset = load_molhiv_dataset(DATASET_ROOT)
labels = np.array([int(d.y.item()) for d in dataset], dtype=np.int64)

print(f"Loaded MolHIV samples: {len(dataset)}")
print(f"Positive ratio: {labels.mean():.4f}")

Loaded MolHIV samples: 41120
Positive ratio: 0.0351


# B-COS Model definition

In [22]:
class Readout(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels=None,
        out_channels=1,
        b=2,
        max_out=1,
        agg: str = "sum",
    ):
        super().__init__()
        if hidden_channels is None:
            self.readout = BcosLinear(in_channels, out_channels, b=b, max_out=max_out)
        else:
            hidden_channels = (
                [hidden_channels]
                if isinstance(hidden_channels, int)
                else hidden_channels
            )
            channels = [in_channels] + hidden_channels + [out_channels]
            self.readout = BcosSequential(
                *[
                    BcosLinear(d_in, d_out, b=b, max_out=max_out)
                    for d_in, d_out in zip(channels[:-1], channels[1:])
                ]
            )
        match agg:
            case "sum":
                self.agg = SumAggregation()
            case _:
                raise ValueError(f"Aggregation '{agg}' not supported.")

    def forward(self, x, batch):
        raise NotImplementedError


class AggThenReadout(Readout):
    def forward(self, x, batch):
        z = self.agg(x, batch)
        out = self.readout(z)
        return out

class ReadoutThenAgg(Readout):
    def forward(self, x, batch):
        z = self.readout(x)
        out = self.agg(z, batch)
        return out



In [23]:
class BcosGINEConv(MessagePassing):
    def __init__(
        self,
        channels: list[int],
        edge_dim: int,
        b: float = 2.0,
        max_out: int = 1,
        eps: float = 0.0,
        train_eps: bool = False,
        **kwargs
    ):
        # We use 'add' aggregation to stay true to the GIN formula
        kwargs.setdefault("aggr", "add")
        super().__init__(**kwargs)
        
        # The MLP part of GIN, using B-cos layers to preserve dynamic linearity
        self.transform = BcosSequential(
            *[
                BcosLinear(din, dout, b=b, max_out=max_out)
                for din, dout in zip(channels[:-1], channels[1:])
            ]
        )
        
        self.initial_eps = eps
        if train_eps:
            self.eps = torch.nn.Parameter(torch.Tensor([eps]))
        else:
            self.register_buffer("eps", torch.Tensor([eps]))

    def forward(self, x, edge_index, edge_attr):
        # edge_attr is expected to be pre-projected to match x dimension
        
        # 1. Propagate messages
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        
        # 2. Combine step: (1 + eps) * center_node + aggregated_messages
        out = (1 + self.eps) * x + out
        
        # 3. Apply the B-cos MLP transformation
        return self.transform(out)

    def message(self, x_j, edge_attr):
        # STRICT LINEARITY: Just add the edge attributes. 
        # No standard ReLUs allowed!
        return x_j + edge_attr

In [29]:
class PureBcosGINE(nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 4,
        num_classes: int = 1,
        b: float = 2.0,
        max_out: int = 1,
        dropout: float = 0.5, 
    ):
        super().__init__()
        
        # 1. Initial Embeddings
        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out)
        
        # 2. Convolutional Layers
        self.convs = nn.ModuleList([
            BcosGINEConv(
                channels=[hidden_dim, hidden_dim], 
                edge_dim=hidden_dim, 
                b=b, 
                max_out=max_out
            ) for _ in range(num_layers)
        ])
        
        # 3. Readout Components (Readout-Then-Aggregate Strategy)
        self.readout_hidden = BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out)
        self.dropout_layer = nn.Dropout(dropout)
        self.readout_out = BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out)
        
        # 4. Aggregation
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = x.float()
        edge_attr = edge_attr.float()
        x = self.lin_node(x)
        e = self.lin_edge(edge_attr)
        
        for conv in self.convs:
            x = conv(x, edge_index, e)
        
        # Readout Then Agg logic
        
        # Step A: Transform node features through the hidden readout
        x = self.readout_hidden(x)
        
        # Step B: Apply dropout to the hidden representations (not the final logits)
        x = self.dropout_layer(x)
        
        # Step C: Project to class logits
        node_logits = self.readout_out(x)
        
        # Step D: Aggregate into graph-level logits
        graph_logits = self.agg(node_logits, batch)
        
        return graph_logits.squeeze(-1)

In [30]:
sample = dataset[0]
device = get_device()
model = PureBcosGINE(
    node_dim=int(sample.x.size(-1)),
    edge_dim=int(sample.edge_attr.size(-1)),
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    b=cfg.b,
    max_out=cfg.max_out,
).to(device)
batch = next(iter(DataLoader([sample], batch_size=1))).to(device)
with torch.no_grad():
    out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
print(out.shape)

torch.Size([1])


In [26]:
def compute_roc_auc_from_logits(logits: List[float], labels: List[int]) -> float:
    y_true = np.asarray(labels, dtype=np.int64)
    y_score = torch.sigmoid(torch.tensor(logits)).numpy()
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def run_epoch(model, loader, optimizer, device):
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss = 0.0
    logits_all, y_all = [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        y = batch.y.view(-1).float()
        loss = F.binary_cross_entropy_with_logits(logits, y)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += float(loss.item()) * y.size(0)
        logits_all.extend(logits.detach().cpu().tolist())
        y_all.extend(batch.y.view(-1).detach().cpu().tolist())

    avg_loss = total_loss / max(1, len(y_all))
    roc_auc = compute_roc_auc_from_logits(logits_all, y_all)
    return avg_loss, roc_auc

def train_one_seed(model_ctor, dataset, train_idx, val_idx, test_idx, cfg: TrainConfig, seed: int, device: torch.device):
    set_seed(seed)
    sample = dataset[0]
    node_dim = int(sample.x.size(-1))
    edge_dim = int(sample.edge_attr.size(-1))

    model = model_ctor(node_dim=node_dim, edge_dim=edge_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader([dataset[i] for i in test_idx], batch_size=cfg.batch_size, shuffle=False)

    best_state = None
    best_val_auc = -float("inf")
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, cfg.max_epochs + 1):
        train_loss, train_auc = run_epoch(model, train_loader, optimizer, device)
        val_loss, val_auc = run_epoch(model, val_loader, optimizer=None, device=device)

        if np.isnan(val_auc):
            val_auc = -float("inf")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Seed {seed} | Epoch {epoch:03d} | train_loss={train_loss:.4f} train_auc={train_auc:.4f} "
                f"| val_loss={val_loss:.4f} val_auc={val_auc:.4f}"
            )

        if bad_epochs >= cfg.early_stop_patience:
            print(f"Seed {seed}: early stop at epoch {epoch} (best val epoch={best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_auc = run_epoch(model, test_loader, optimizer=None, device=device)
    return {
        "seed": int(seed),
        "best_val_epoch": int(best_epoch),
        "best_val_roc_auc": float(best_val_auc),
        "test_roc_auc": float(test_auc),
        "test_loss": float(test_loss),
    }

def summarize_seed_results(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    summary = pd.DataFrame(
        [
            {
                "model": model_name,
                "n_seeds": int(len(df)),
                "test_roc_auc_mean": float(df["test_roc_auc"].mean()),
                "test_roc_auc_std": float(df["test_roc_auc"].std(ddof=1)) if len(df) > 1 else 0.0,
                "best_val_epoch_mean": float(df["best_val_epoch"].mean()),
                "best_val_epoch_std": float(df["best_val_epoch"].std(ddof=1)) if len(df) > 1 else 0.0,
            }
        ]
    )
    return summary

def run_three_seed_experiment(model_name: str, model_ctor, dataset, labels, cfg: TrainConfig, seeds: Iterable[int]):
    device = get_device()
    print(f"\nRunning {model_name} on device: {device}")
    results = []

    for seed in seeds:
        train_idx, val_idx, test_idx = stratified_split_indices(
            labels=labels,
            seed=int(seed),
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
        )
        out = train_one_seed(
            model_ctor=model_ctor,
            dataset=dataset,
            train_idx=train_idx,
            val_idx=val_idx,
            test_idx=test_idx,
            cfg=cfg,
            seed=int(seed),
            device=device,
        )
        results.append(out)
        print(
            f"{model_name} | Seed {seed} | test_auc={out['test_roc_auc']:.4f} "
            f"| best_val_epoch={out['best_val_epoch']} | best_val_auc={out['best_val_roc_auc']:.4f}"
        )

    df = pd.DataFrame(results).sort_values("seed").reset_index(drop=True)
    summary = summarize_seed_results(df, model_name=model_name)
    return df, summary

## B-COS GINE (3 seeds)

In [31]:
bcos_ctor = lambda node_dim, edge_dim: PureBcosGINE(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
    b=cfg.b,
    max_out=cfg.max_out,
)

bcos_runs, bcos_summary = run_three_seed_experiment(
    model_name="B-COS GINE",
    model_ctor=bcos_ctor,
    dataset=dataset,
    labels=labels,
    cfg=cfg,
    seeds=SEEDS,
)

display(bcos_runs)
display(bcos_summary)


Running B-COS GINE on device: cpu
Seed 0 | Epoch 001 | train_loss=0.2237 train_auc=0.4160 | val_loss=0.2392 val_auc=0.4295
Seed 0 | Epoch 001 | train_loss=0.2237 train_auc=0.4160 | val_loss=0.2392 val_auc=0.4295
Seed 0 | Epoch 010 | train_loss=0.1878 train_auc=0.5365 | val_loss=0.1993 val_auc=0.5292
Seed 0 | Epoch 010 | train_loss=0.1878 train_auc=0.5365 | val_loss=0.1993 val_auc=0.5292
Seed 0 | Epoch 020 | train_loss=0.1761 train_auc=0.5853 | val_loss=0.2003 val_auc=0.5491
Seed 0 | Epoch 020 | train_loss=0.1761 train_auc=0.5853 | val_loss=0.2003 val_auc=0.5491
Seed 0 | Epoch 030 | train_loss=0.1659 train_auc=0.6382 | val_loss=0.1924 val_auc=0.5588
Seed 0 | Epoch 030 | train_loss=0.1659 train_auc=0.6382 | val_loss=0.1924 val_auc=0.5588
Seed 0 | Epoch 040 | train_loss=0.1624 train_auc=0.6584 | val_loss=0.2100 val_auc=0.5628
Seed 0 | Epoch 040 | train_loss=0.1624 train_auc=0.6584 | val_loss=0.2100 val_auc=0.5628
Seed 0 | Epoch 050 | train_loss=0.1571 train_auc=0.6806 | val_loss=0.1896 v

,seed,best_val_epoch,best_val_roc_auc,test_roc_auc,test_loss
0,0,117,0.647844,0.701429,0.181917
1,1,110,0.728803,0.747257,0.160901
2,2,120,0.736879,0.722762,0.161398


,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,B-COS GINE,3,0.723816,0.022932,115.666667,5.131601


## Vanilla GINE (3 seeds)

In [8]:
vanilla_ctor = lambda node_dim, edge_dim: VanillaGINE(
    node_dim=node_dim,
    edge_dim=edge_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
)

vanilla_runs, vanilla_summary = run_three_seed_experiment(
    model_name="Vanilla GINE",
    model_ctor=vanilla_ctor,
    dataset=dataset,
    labels=labels,
    cfg=cfg,
    seeds=SEEDS,
)

display(vanilla_runs)
display(vanilla_summary)


Running Vanilla GINE on device: cpu
Seed 0 | Epoch 001 | train_loss=0.3188 train_auc=0.4479 | val_loss=0.2276 val_auc=0.4076
Seed 0 | Epoch 001 | train_loss=0.3188 train_auc=0.4479 | val_loss=0.2276 val_auc=0.4076
Seed 0 | Epoch 010 | train_loss=0.1461 train_auc=0.6786 | val_loss=0.2189 val_auc=0.5468
Seed 0 | Epoch 010 | train_loss=0.1461 train_auc=0.6786 | val_loss=0.2189 val_auc=0.5468
Seed 0 | Epoch 020 | train_loss=0.1271 train_auc=0.7607 | val_loss=0.1395 val_auc=0.7144
Seed 0 | Epoch 020 | train_loss=0.1271 train_auc=0.7607 | val_loss=0.1395 val_auc=0.7144
Seed 0 | Epoch 030 | train_loss=0.1191 train_auc=0.7914 | val_loss=0.2076 val_auc=0.7088
Seed 0 | Epoch 030 | train_loss=0.1191 train_auc=0.7914 | val_loss=0.2076 val_auc=0.7088
Seed 0 | Epoch 040 | train_loss=0.1119 train_auc=0.8307 | val_loss=0.1333 val_auc=0.7464
Seed 0 | Epoch 040 | train_loss=0.1119 train_auc=0.8307 | val_loss=0.1333 val_auc=0.7464
Seed 0 | Epoch 050 | train_loss=0.1093 train_auc=0.8434 | val_loss=0.1261

,seed,best_val_epoch,best_val_roc_auc,test_roc_auc,test_loss
0,0,63,0.782346,0.800641,0.113336
1,1,114,0.835505,0.780884,0.131119
2,2,99,0.842812,0.786474,0.122661


,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,Vanilla GINE,3,0.789333,0.010184,92.0,26.210685


## Comparative Report

In [9]:
comparison = pd.concat([bcos_summary, vanilla_summary], ignore_index=True)

comparison = comparison[[
    "model",
    "n_seeds",
    "test_roc_auc_mean",
    "test_roc_auc_std",
    "best_val_epoch_mean",
    "best_val_epoch_std",
]]

display(comparison)

print("\nCompact report:")
for _, row in comparison.iterrows():
    print(
        f"{row['model']}: test ROC-AUC = {row['test_roc_auc_mean']:.4f} ± {row['test_roc_auc_std']:.4f}; "
        f"best val epoch = {row['best_val_epoch_mean']:.2f} ± {row['best_val_epoch_std']:.2f}"
    )

,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,B-COS GINE,3,0.806814,0.023807,72.333333,36.665151
1,Vanilla GINE,3,0.789333,0.010184,92.000000,26.210685



Compact report:
B-COS GINE: test ROC-AUC = 0.8068 ± 0.0238; best val epoch = 72.33 ± 36.67
Vanilla GINE: test ROC-AUC = 0.7893 ± 0.0102; best val epoch = 92.00 ± 26.21
